# MURA X-Ray Classification

**Author:** Georgios Kitsakis  
**Institution:** Athens University of Economics and Business (AUEB)

---

## What are we doing?

We have X-ray images of bones (shoulder, elbow, wrist, hand, finger, forearm, humerus) and we want to automatically decide if each image is **normal** or **abnormal**.

This is a **binary classification** problem. The model outputs a number between 0 and 1:
- Close to **0** the model thinks the X-ray is **normal**
- Close to **1** the model thinks the X-ray is **abnormal**

## Dataset: MURA

MURA (MUsculoskeletal RAdiographs) was created by Stanford University.
- ~40,000 X-ray images across 7 body parts
- Already split into **train** (~36,800 images) and **validation** (~3,200 images)
- Each image is labeled by a radiologist as normal or abnormal

## Models we will train

| # | Model | Type |
|---|---|---|
| 1 | Custom CNN | Built from scratch |
| 2 | DenseNet121 | Pre-trained on ImageNet, fine-tuned |
| 3 | EfficientNetB0 | Pre-trained on ImageNet, fine-tuned |
| 4 | CNN-GRU Hybrid | CNN + recurrent network combined |

---
## Download the Dataset

In [ ]:
import os
import getpass

# Paste your Kaggle API token when prompted (the KGAT_... one from your Kaggle settings)
token = getpass.getpass('Kaggle API token: ')
os.environ['KAGGLE_TOKEN'] = token

!kaggle datasets download -d cjinny/mura-v11 -p /content/mura
!unzip -q /content/mura/mura-v11.zip -d /content/mura

DATASET_PATH = '/content/mura'
print('Done! Dataset path exists:', os.path.exists(DATASET_PATH))

---
## Imports & Configuration

In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import backend as K
from tensorflow.keras.applications import DenseNet121, EfficientNetB0
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (Input, Conv2D, Dense, Dropout, MaxPool2D,
                                      BatchNormalization, GlobalAveragePooling2D,
                                      GRU, TimeDistributed, Masking)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

%matplotlib inline

IMG_SIZE   = 224   # resize all images to 224x224 pixels
BATCH_SIZE = 32    # number of images processed at once
EPOCHS     = 30    # maximum training epochs (EarlyStopping will stop earlier)
MAX_VIEWS  = 7     # max X-ray views per study (used by the CNN-GRU sequence generator)

print('TensorFlow version:', tf.__version__)

KeyboardInterrupt: 

---
## Load the Data

The MURA dataset comes with CSV files listing every image path and its label.
We read those CSVs and build a single DataFrame with columns: `image_path`, `label_type`, `set_type`, `category`.

In [ ]:
def extract_set_category(path):
    pattern = r'.*(?P<set_type>train|valid)/(?P<category>XR_[A-Z]+)/(?P<patient_id>patient\d+)/study.*'
    match = re.match(pattern, path)
    return match.groupdict() if match else None


def load_split(split, dataset_path=DATASET_PATH):
    image_paths = pd.read_csv(f'{dataset_path}/MURA-v1.1/{split}_image_paths.csv',
                              header=None, names=['image_path'])
    image_paths['path'] = image_paths['image_path'].apply(
        lambda x: '/'.join(x.split('/')[:-1]) + '/'
    )
    labels = pd.read_csv(f'{dataset_path}/MURA-v1.1/{split}_labeled_studies.csv',
                         header=None, names=['path', 'label'])
    return labels.merge(image_paths, on='path', how='left')


df = pd.concat([load_split('train'), load_split('valid')]).reset_index(drop=True)
df = pd.concat([df, pd.DataFrame(df['path'].apply(extract_set_category).tolist())], axis=1)
df['label_type'] = df['label'].map({1: 'abnormal', 0: 'normal'})
df['image_path'] = df['image_path'].apply(lambda x: os.path.join(DATASET_PATH, x))

print(f'Total images: {len(df)}')
print(f'Train: {len(df[df["set_type"]=="train"])}  |  Validation: {len(df[df["set_type"]=="valid"])}')
df.head()

---
## Explore the Dataset

In [ ]:
print('Label distribution:')
print(df['label_type'].value_counts())
print()
print('Images per body part:')
print(df.groupby(['category', 'label_type']).size().unstack(fill_value=0))

> **Important:** There are more normal images (~23,600) than abnormal (~16,400). This **class imbalance** means a naive model could get ~60% accuracy just by always predicting 'normal'. We fix this with **class weights** - the model is penalised more heavily for mistakes on the minority class.

In [ ]:
def show_samples(dataframe, title=''):
    sample = dataframe.sample(12, random_state=42)
    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    for ax, (_, row) in zip(axes.flat, sample.iterrows()):
        img = load_img(row['image_path'], target_size=(IMG_SIZE, IMG_SIZE))
        ax.imshow(img)
        ax.set_title(f"{row['category'].replace('XR_', '')}\n{row['label_type']}", fontsize=8)
        ax.axis('off')
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_samples(df[df['label_type'] == 'normal'],   title='Normal X-Rays')
show_samples(df[df['label_type'] == 'abnormal'], title='Abnormal X-Rays')

---
## Data Generators

We use Keras `ImageDataGenerator` to:
- Load images from disk in batches (avoids loading 40k images into RAM at once)
- Apply **data augmentation** on training images (random flips, rotations, zooms) to make the model more robust
- Scale pixel values from [0, 255] to [0, 1]

In [ ]:
df_train = df[df['set_type'] == 'train'].reset_index(drop=True)
df_valid = df[df['set_type'] == 'valid'].reset_index(drop=True)

# Class weights to compensate for the normal/abnormal imbalance
n_normal   = len(df_train[df_train['label_type'] == 'normal'])
n_abnormal = len(df_train[df_train['label_type'] == 'abnormal'])
total      = n_normal + n_abnormal
class_weight = {
    0: total / (2 * n_normal),    # normal
    1: total / (2 * n_abnormal),  # abnormal gets higher weight (minority class)
}
print('Class weights:', class_weight)

# Standard generators (rescale to [0,1]) - used by Custom CNN and CNN-GRU
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    df_train, x_col='image_path', y_col='label_type',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=['normal', 'abnormal'],
    shuffle=True, seed=42
)
val_gen = val_datagen.flow_from_dataframe(
    df_valid, x_col='image_path', y_col='label_type',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=['normal', 'abnormal'],
    shuffle=False
)

print(f'Train batches: {len(train_gen)}  |  Val batches: {len(val_gen)}')

---
## Helper Functions

Reusable functions for plotting and evaluating — shared across all models.

In [ ]:
def plot_history(history, model_name=''):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Training History - {model_name}', fontsize=13, fontweight='bold')
    for ax, metric in zip(axes, ['accuracy', 'loss']):
        ax.plot(history.history[metric],          label='Train')
        ax.plot(history.history[f'val_{metric}'], label='Validation')
        ax.set_title(metric.capitalize())
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def evaluate(model, generator, model_name=''):
    loss, acc = model.evaluate(generator, verbose=0)
    print(f'{model_name} - Loss: {loss:.4f} | Accuracy: {acc:.4f}')
    return loss, acc


def plot_confusion_matrix(model, generator, model_name=''):
    generator.reset()
    y_pred = (model.predict(generator, verbose=0) > 0.5).astype(int).flatten()
    y_true = generator.classes
    labels = list(generator.class_indices.keys())
    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix - {model_name}', fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(classification_report(y_true, y_pred, target_names=labels))


def show_misclassified(model, generator, model_name=''):
    generator.reset()
    y_pred = (model.predict(generator, verbose=0) > 0.5).astype(int).flatten()
    y_true = generator.classes
    labels = list(generator.class_indices.keys())
    wrong  = np.where(y_pred != y_true)[0]
    sample = np.random.choice(wrong, min(12, len(wrong)), replace=False)
    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    for ax, idx in zip(axes.flat, sample):
        img = load_img(generator.filenames[idx], target_size=(IMG_SIZE, IMG_SIZE))
        ax.imshow(img)
        ax.set_title(f"True: {labels[y_true[idx]]}\nPred: {labels[y_pred[idx]]}", fontsize=8, color='red')
        ax.axis('off')
    plt.suptitle(f'Misclassified - {model_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f'Total wrong: {len(wrong)} / {len(y_true)}')

---
## Model 1 - Custom CNN

The first model is built **entirely from scratch** — no pre-trained weights, no external knowledge. It learns everything about X-rays from the MURA training data alone.

### Architecture

```
Input image (224 x 224 x 3)
  |
Conv2D(32)  -> BatchNorm -> MaxPool -> Dropout(0.25)
Conv2D(64)  -> BatchNorm -> MaxPool -> Dropout(0.25)
Conv2D(128) -> BatchNorm -> MaxPool -> Dropout(0.25)
Conv2D(256) -> BatchNorm -> MaxPool -> Dropout(0.25)
  |
GlobalAveragePooling  (collapses spatial dims into 1 number per filter)
  |
Dense(256) -> Dropout(0.4)
  |
Dense(1, sigmoid)  ->  probability of being abnormal
```

- Each conv block doubles the filters (32->64->128->256) — learns increasingly complex patterns
- **BatchNorm** keeps activations stable, speeds up training
- **Dropout** randomly disables neurons — reduces overfitting
- **GlobalAveragePooling** uses far fewer parameters than Flatten

In [ ]:
def build_custom_cnn():
    inp = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x   = inp
    for filters in [32, 64, 128, 256]:
        x = Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
        x = BatchNormalization()(x)
        x = MaxPool2D((2, 2))(x)
        x = Dropout(0.25)(x)
    x   = GlobalAveragePooling2D()(x)
    x   = Dense(256, activation='relu')(x)
    x   = Dropout(0.4)(x)
    out = Dense(1, activation='sigmoid')(x)
    return Model(inputs=inp, outputs=out)


cnn_model = build_custom_cnn()
cnn_model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
cnn_model.summary()

In [ ]:
es  = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

cnn_history = cnn_model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS, callbacks=[es, rlr],
    class_weight=class_weight, verbose=1
)

plot_history(cnn_history, model_name='Custom CNN')
cnn_eval = evaluate(cnn_model, val_gen, model_name='Custom CNN')
plot_confusion_matrix(cnn_model, val_gen, model_name='Custom CNN')
show_misclassified(cnn_model, val_gen, model_name='Custom CNN')

---
## Model 2 - DenseNet121 (Transfer Learning)

Instead of training from scratch, we reuse a network already trained on **ImageNet** (1.2M natural images). The model already knows how to detect edges, textures, and shapes — we just fine-tune it for X-rays.

**Why DenseNet121?** Stanford used it in their original MURA paper. It connects every layer to every previous layer (dense connections), giving excellent gradient flow and strong results on medical images.

### Two-phase training

**Freeze the base, train only the new head (5 epochs)**  
This warms up the head without touching the ImageNet weights.

**Unfreeze everything, fine-tune with lr=1e-5**  
The tiny learning rate gently adjusts all weights without destroying what was learned on ImageNet.

In [ ]:
# DenseNet requires its own pixel normalisation - NOT a simple /255
dn_train_datagen = ImageDataGenerator(
    preprocessing_function=densenet_preprocess,
    horizontal_flip=True, rotation_range=15, zoom_range=0.1,
    width_shift_range=0.1, height_shift_range=0.1
)
dn_val_datagen = ImageDataGenerator(preprocessing_function=densenet_preprocess)

dn_train_gen = dn_train_datagen.flow_from_dataframe(
    df_train, x_col='image_path', y_col='label_type',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=['normal', 'abnormal'], shuffle=True, seed=42
)
dn_val_gen = dn_val_datagen.flow_from_dataframe(
    df_valid, x_col='image_path', y_col='label_type',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=['normal', 'abnormal'], shuffle=False
)

In [ ]:
dn_base  = DenseNet121(include_top=False, weights='imagenet',
                       input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling='avg')
x        = Dense(256, activation='relu')(dn_base.output)
x        = Dropout(0.4)(x)
dn_out   = Dense(1, activation='sigmoid')(x)
dn_model = Model(inputs=dn_base.input, outputs=dn_out)

# Phase 1: freeze the base - only the head trains
for layer in dn_base.layers:
    layer.trainable = False

dn_model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
es = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1)

print('Phase 1: training the head (base frozen) ...')
dn_p1 = dn_model.fit(
    dn_train_gen, validation_data=dn_val_gen,
    epochs=5, callbacks=[es], class_weight=class_weight, verbose=1
)
print(f'Phase 1 done ({len(dn_p1.history["loss"])} epochs)')

In [ ]:
# Phase 2: unfreeze everything and fine-tune with a tiny learning rate
for layer in dn_base.layers:
    layer.trainable = True

dn_model.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
es  = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)

print('Phase 2: fine-tuning the full network ...')
dn_p2 = dn_model.fit(
    dn_train_gen, validation_data=dn_val_gen,
    epochs=EPOCHS, callbacks=[es, rlr], class_weight=class_weight, verbose=1
)
print(f'Phase 2 done ({len(dn_p2.history["loss"])} epochs)')

plot_history(dn_p2, model_name='DenseNet121 - Fine-tune')
dn_eval = evaluate(dn_model, dn_val_gen, model_name='DenseNet121')
plot_confusion_matrix(dn_model, dn_val_gen, model_name='DenseNet121')
show_misclassified(dn_model, dn_val_gen, model_name='DenseNet121')

---
## Model 3 - EfficientNetB0 (Transfer Learning)

A second pre-trained model for comparison with DenseNet121.

**What makes EfficientNet different?**  
Traditional CNNs scale by going deeper OR wider. EfficientNet uses **compound scaling** — it scales depth, width, and image resolution together in a balanced ratio, achieving better accuracy with fewer parameters.

**Why compare it to DenseNet?**  
DenseNet was specifically tested on MURA. EfficientNet is a more general modern architecture. Comparing them lets us see whether the task benefits from a medically-motivated design.

Same two-phase training approach as DenseNet121.

In [ ]:
# EfficientNet also has its own preprocessing - different from DenseNet's
eff_train_datagen = ImageDataGenerator(
    preprocessing_function=eff_preprocess,
    horizontal_flip=True, rotation_range=15, zoom_range=0.1,
    width_shift_range=0.1, height_shift_range=0.1
)
eff_val_datagen = ImageDataGenerator(preprocessing_function=eff_preprocess)

eff_train_gen = eff_train_datagen.flow_from_dataframe(
    df_train, x_col='image_path', y_col='label_type',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=['normal', 'abnormal'], shuffle=True, seed=42
)
eff_val_gen = eff_val_datagen.flow_from_dataframe(
    df_valid, x_col='image_path', y_col='label_type',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', classes=['normal', 'abnormal'], shuffle=False
)

In [ ]:
eff_base  = EfficientNetB0(include_top=False, weights='imagenet',
                           input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling='avg')
x         = Dense(256, activation='relu')(eff_base.output)
x         = Dropout(0.4)(x)
eff_out   = Dense(1, activation='sigmoid')(x)
eff_model = Model(inputs=eff_base.input, outputs=eff_out)

# Phase 1: frozen base
for layer in eff_base.layers:
    layer.trainable = False

eff_model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
es = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1)

print('Phase 1: training the head (base frozen) ...')
eff_p1 = eff_model.fit(
    eff_train_gen, validation_data=eff_val_gen,
    epochs=5, callbacks=[es], class_weight=class_weight, verbose=1
)
print(f'Phase 1 done ({len(eff_p1.history["loss"])} epochs)')

In [ ]:
# Phase 2: full fine-tune
for layer in eff_base.layers:
    layer.trainable = True

eff_model.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
es  = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)

print('Phase 2: fine-tuning the full network ...')
eff_p2 = eff_model.fit(
    eff_train_gen, validation_data=eff_val_gen,
    epochs=EPOCHS, callbacks=[es, rlr], class_weight=class_weight, verbose=1
)
print(f'Phase 2 done ({len(eff_p2.history["loss"])} epochs)')

plot_history(eff_p2, model_name='EfficientNetB0 - Fine-tune')
eff_eval = evaluate(eff_model, eff_val_gen, model_name='EfficientNetB0')
plot_confusion_matrix(eff_model, eff_val_gen, model_name='EfficientNetB0')
show_misclassified(eff_model, eff_val_gen, model_name='EfficientNetB0')

---
## Model 4 - CNN-GRU (Study-Level Sequence Model)

An experimental architecture that treats each patient study as a **sequence of X-ray views**.

**The idea:**  
Each MURA study contains multiple images of the same body part from different angles. Instead of classifying images one by one and voting, we let the model read all views together — the GRU sees the full picture before deciding.

### How it works

```
Study: [view1, view2, view3, ..., viewN]  (padded to MAX_VIEWS=7)
  |
EfficientNetB0 (frozen, shared weights) applied to each view independently
  → feature vector per view  (shape: 1280)
  |
GRU reads the sequence of feature vectors
  → single summary hidden state
  |
Dense(1, sigmoid)  →  normal / abnormal
```

- **`TimeDistributed`** applies the same CNN to every view in the sequence
- **`Masking`** tells the GRU to ignore zero-padded slots (studies with fewer than 7 images)
- The CNN backbone is **frozen** — we only train the GRU and the final Dense layers

In [ ]:
class StudySequenceGenerator(tf.keras.utils.Sequence):
    """
    Custom generator that yields one study (patient visit) at a time as a sequence of images.

    Each study = multiple X-ray views of the same body part, all sharing one label.
    Images are sorted alphabetically within each study for consistent ordering.
    Studies shorter than MAX_VIEWS are zero-padded; longer ones are truncated.
    """

    def __init__(self, df, batch_size=8, max_views=MAX_VIEWS, augment=False, shuffle=True):
        # Group rows by study folder path — each group is one study
        self.studies    = df.groupby('path')
        self.study_ids  = list(self.studies.groups.keys())
        self.batch_size = batch_size
        self.max_views  = max_views
        self.augment    = augment
        self.shuffle    = shuffle
        if shuffle:
            np.random.shuffle(self.study_ids)

    def __len__(self):
        return int(np.ceil(len(self.study_ids) / self.batch_size))

    def __getitem__(self, idx):
        batch_ids = self.study_ids[idx * self.batch_size:(idx + 1) * self.batch_size]

        # X: (batch, MAX_VIEWS, H, W, C) — zero-initialised, padding is left as zeros
        X = np.zeros((len(batch_ids), self.max_views, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        y = np.zeros(len(batch_ids), dtype=np.float32)

        for i, study_path in enumerate(batch_ids):
            group = self.studies.get_group(study_path)

            # Sort filenames so the view order is the same every time
            image_paths = sorted(group['image_path'].tolist())[:self.max_views]

            for j, img_path in enumerate(image_paths):
                img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE), color_mode='rgb')
                arr = img_to_array(img) / 255.0

                if self.augment:
                    if np.random.rand() > 0.5:
                        arr = arr[:, ::-1, :]   # random horizontal flip
                    angle = np.random.uniform(-15, 15)
                    arr = tf.keras.preprocessing.image.apply_affine_transform(arr, theta=angle)

                X[i, j] = arr

            y[i] = group['label'].iloc[0]

        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.study_ids)


train_seq = StudySequenceGenerator(df_train, batch_size=8, augment=True,  shuffle=True)
val_seq   = StudySequenceGenerator(df_valid, batch_size=8, augment=False, shuffle=False)

print('Train studies:', len(train_seq.study_ids))
print('Val   studies:', len(val_seq.study_ids))

In [ ]:
def build_cnn_gru():
    # Frozen EfficientNetB0 as the feature extractor — one forward pass per image view
    backbone = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling='avg',      # outputs a 1280-dim vector per image
    )
    backbone.trainable = False  # freeze all backbone weights

    # Input: a full study = sequence of up to MAX_VIEWS images
    inp = Input(shape=(MAX_VIEWS, IMG_SIZE, IMG_SIZE, 3))

    # Apply the same backbone to each view independently → (batch, MAX_VIEWS, 1280)
    x = TimeDistributed(backbone)(inp)

    # Tell the GRU to skip zero-padded slots (studies with fewer than MAX_VIEWS images)
    x = Masking(mask_value=0.0)(x)

    # GRU reads the sequence of feature vectors, returns one final summary vector
    x = GRU(256, return_sequences=False, dropout=0.3)(x)

    x   = Dense(128, activation='relu')(x)
    x   = Dropout(0.3)(x)
    out = Dense(1, activation='sigmoid')(x)

    return Model(inputs=inp, outputs=out)


gru_model = build_cnn_gru()
gru_model.compile(
    optimizer=Adam(1e-4),   # lower LR — GRU is more sensitive than a plain CNN
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
gru_model.summary()

es  = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)

gru_history = gru_model.fit(
    train_seq, validation_data=val_seq,
    epochs=EPOCHS, callbacks=[es, rlr],
)

plot_history(gru_history, model_name='CNN-GRU')

# Evaluation helpers for sequence models (generator yields X,y tuples, not Keras generator objects)
def evaluate_seq(model, generator, model_name=''):
    loss, acc = model.evaluate(generator, verbose=0)
    print(f'{model_name} - Loss: {loss:.4f} | Accuracy: {acc:.4f}')
    return loss, acc

def plot_confusion_matrix_seq(model, generator, model_name=''):
    y_true, y_pred = [], []
    for i in range(len(generator)):
        X_batch, y_batch = generator[i]
        preds = (model.predict(X_batch, verbose=0) > 0.5).astype(int).flatten()
        y_true.extend(y_batch.astype(int).tolist())
        y_pred.extend(preds.tolist())
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    labels = ['normal', 'abnormal']
    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix - {model_name}', fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(classification_report(y_true, y_pred, target_names=labels))

gru_eval = evaluate_seq(gru_model, val_seq, model_name='CNN-GRU')
plot_confusion_matrix_seq(gru_model, val_seq, model_name='CNN-GRU')

---
## Final Comparison

All 4 models evaluated on the same MURA validation set.

In [ ]:
print(f"{'Model':<35} {'Val Accuracy':>14} {'Val Loss':>12}")
print('-' * 63)
for name, (loss, acc) in [
    ('Custom CNN (from scratch)',         cnn_eval),
    ('DenseNet121 (transfer learning)',   dn_eval),
    ('EfficientNetB0 (transfer learning)',eff_eval),
    ('CNN-GRU (study-level sequence)',    gru_eval),
]:
    print(f'{name:<35} {acc:>14.4f} {loss:>12.4f}')